In [1]:
import sys
import subprocess
try:
    from econml.dml import CausalForestDML
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "econml", "openpyxl"])
    from econml.dml import CausalForestDML

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


print("=" * 80)
print("1. VERİ YÜKLEME")
print("=" * 80)

os.makedirs("/content/outputs", exist_ok=True)

df = pd.read_csv("/content/tekstil_karbon_datasetM.csv")
print(f"Orijinal veri seti boyutu: {df.shape}")

df = df.drop(columns=["batch_id"], errors="ignore")

if "ay" in df.columns:
    df["ay"] = df["ay"].astype(str)

missing_total = df.isnull().sum().sum()
print(f"Toplam eksik veri sayısı: {missing_total}")


print("\n" + "=" * 80)
print("2. TREATMENT VE OUTCOME TANIMLARI")
print("=" * 80)

ALL_TREATMENTS = [
    "t_yenilenebilir_enerji",
    "t_yerel_tedarik",
    "t_dusuk_em_boya",
    "t_geri_donusum_prog",
    "t_makine_modernizasyon",
    "t_parti_optimizasyon",
]

OUTCOMES = [
    "scope2_kgCO2e",
    "scope3_tasima_kgCO2e",
    "scope3_atik_kgCO2e",
    "scope3_kimyasal_kgCO2e",
]

CATEGORICALS = ["urun_tipi", "kumas_tipi", "tasima_modu", "ay"]


SEGMENT_VARS = {
    "urun_tipi"            : "cat",
    "kumas_tipi"           : "cat",
    "kapasite_kullanim_pct": "qcut",
    "makine_yasi_yil"      : "qcut",
    "parti_buyuklugu"      : "qcut",
}


COLORS = {
    "primary"    : "#2563EB",
    "secondary"  : "#10B981",
    "neutral"    : "#6B7280",
    "bg"         : "#F8FAFC",
    "placebo"    : "#9CA3AF",
    "sensitivity": "#7C3AED",
    "high_effect": "#059669",
    "low_effect" : "#DC2626",
}

labels_short = {
    "t_yenilenebilir_enerji" : "Yenilenebilir\nEnerji",
    "t_makine_modernizasyon" : "Makine\nModernizasyon",
    "t_yerel_tedarik"        : "Yerel\nTedarik",
    "t_geri_donusum_prog"    : "Geri\nDönüşüm",
    "t_dusuk_em_boya"        : "Düşük Em.\nBoya",
    "t_parti_optimizasyon"   : "Parti\nOptimizasyon",
}

BASE_CONFOUNDER_SETS = {
    "t_yenilenebilir_enerji": {
        "outcome": "scope2_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "kumas_gramaj_g_m2", "uretim_suresi_dk_adet",
            "vardiya_sayisi", "kapasite_kullanim_pct",
            "makine_yasi_yil", "ay",
        ],
        "mediators": ["elektrik_kwh"],
        "label": "Yenilenebilir Enerji → Scope 2",
    },
    "t_makine_modernizasyon": {
        "outcome": "scope2_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "kumas_gramaj_g_m2", "uretim_suresi_dk_adet",
            "vardiya_sayisi", "kapasite_kullanim_pct",
            "makine_yasi_yil", "ay",
        ],
        "mediators": ["elektrik_kwh"],
        "label": "Makine Modernizasyon → Scope 2",
    },
    "t_yerel_tedarik": {
        "outcome": "scope3_tasima_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "hammadde_ton", "tasima_modu", "ay",
        ],
        "mediators": ["hammadde_mesafe_km"],
        "label": "Yerel Tedarik → Scope 3 Taşıma",
    },
    "t_geri_donusum_prog": {
        "outcome": "scope3_atik_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "hammadde_ton", "kumas_gramaj_g_m2",
            "kapasite_kullanim_pct", "ay",
        ],
        "mediators": ["tekstil_atik_kg"],
        "label": "Geri Dönüşüm Programı → Scope 3 Atık",
    },
    "t_dusuk_em_boya": {
        "outcome": "scope3_kimyasal_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "kumas_gramaj_g_m2", "su_tuketim_lt_adet",
            "uretim_suresi_dk_adet", "ay",
        ],
        "mediators": ["kimyasal_kg_adet"],
        "label": "Düşük Emisyonlu Boya → Scope 3 Kimyasal",
    },
    "t_parti_optimizasyon": {
        "outcome": "scope2_kgCO2e",
        "base_confounders": [
            "urun_tipi", "kumas_tipi", "parti_buyuklugu",
            "hammadde_ton", "kumas_gramaj_g_m2",
            "vardiya_sayisi", "kapasite_kullanim_pct",
            "makine_yasi_yil", "ay",
        ],
        "mediators": [
            "uretim_suresi_dk_adet", "elektrik_kwh",
            "tekstil_atik_kg", "atiksu_lt",
        ],
        "label": "Parti Optimizasyon → Scope 2",
    },
}


CONFOUNDER_SETS = {}
for t, cfg in BASE_CONFOUNDER_SETS.items():
    other_treatments = [ot for ot in ALL_TREATMENTS if ot != t]
    CONFOUNDER_SETS[t] = {
        "outcome"                       : cfg["outcome"],
        "confounders"                   : cfg["base_confounders"] + other_treatments,
        "base_confounders"              : cfg["base_confounders"],
        "other_treatments_as_confounders": other_treatments,
        "mediators"                     : cfg["mediators"],
        "label"                         : cfg["label"],
    }

print("Confounder setleri oluşturuldu (temel + eş zamanlı treatmentlar).")


print("\n" + "=" * 80)
print("3. SÜTUN KONTROLÜ")
print("=" * 80)

required_cols = set(ALL_TREATMENTS)
for cfg in CONFOUNDER_SETS.values():
    required_cols.add(cfg["outcome"])
    required_cols.update(cfg["confounders"])
    required_cols.update(cfg["mediators"])

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print("Veri setindeki mevcut sütunlar:")
    print(df.columns.tolist())
    raise ValueError(f"Veri setinde eksik sütunlar var: {missing_cols}")
else:
    print("Tüm gerekli sütunlar veri setinde mevcut.")


def prepare_X(data, confounders):
    X = data[confounders].copy()
    cats = [c for c in confounders if c in CATEGORICALS]
    if cats:
        X = pd.get_dummies(X, columns=cats, drop_first=False)
    return X.astype(float)


def run_causal_forest(data, treatment_col, config, n_estimators=500, seed=42):
    outcome_col = config["outcome"]
    confounders = config["confounders"]

    Y = data[outcome_col].values
    W = data[treatment_col].values.astype(int)
    X = prepare_X(data, confounders)

    model_y = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=10, random_state=seed, n_jobs=-1
    )
    model_t = RandomForestClassifier(
        n_estimators=200, min_samples_leaf=10, random_state=seed, n_jobs=-1
    )

    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        n_estimators=n_estimators,
        min_samples_leaf=10,
        max_depth=None,
        random_state=seed,
        inference=True,
        discrete_treatment=True,
        cv=5,
    )
    cf.fit(Y, W, X=X)

    ate    = float(np.atleast_1d(cf.ate(X))[0])
    ate_ci = cf.ate_interval(X, alpha=0.05)
    ate_lb = float(np.atleast_1d(ate_ci[0])[0])
    ate_ub = float(np.atleast_1d(ate_ci[1])[0])

    ite          = cf.effect(X)
    ite_lb, ite_ub = cf.effect_interval(X, alpha=0.05)
    feat_imp     = cf.feature_importances_
    feat_names   = X.columns.tolist()

    return {
        "model"    : cf,
        "X"        : X,
        "Y"        : Y,
        "W"        : W,
        "ate"      : ate,
        "ate_lb"   : ate_lb,
        "ate_ub"   : ate_ub,
        "ite"      : ite,
        "ite_lb"   : ite_lb,
        "ite_ub"   : ite_ub,
        "feat_imp" : dict(zip(feat_names, feat_imp)),
        "n_treated": int(W.sum()),
        "n_control": int((W == 0).sum()),
    }


def make_segment_col(series, mode, label):
    if mode == "qcut":
        try:
            return pd.qcut(
                series, q=3,
                labels=[f"{label}_Düşük", f"{label}_Orta", f"{label}_Yüksek"],
                duplicates="drop",
            ).astype(str)
        except Exception:
            return series.astype(str)
    else:
        return series.astype(str)


def cate_segment_analysis(df_src, ite_array, treatment_col, config):
    work = df_src.copy()
    work["_ite"] = ite_array
    outcome_col  = config["outcome"]


    seg_rows = []
    for var, mode in SEGMENT_VARS.items():
        if var not in work.columns:
            continue
        seg_col = make_segment_col(work[var], mode, var)
        for grp_val, grp_df in work.groupby(seg_col):
            seg_rows.append({
                "Treatment"       : treatment_col,
                "Segment_Degisken": var,
                "Segment_Grubu"   : str(grp_val),
                "N"               : len(grp_df),
                "CATE_Ortalama"   : round(grp_df["_ite"].mean(), 4),
                "CATE_Medyan"     : round(grp_df["_ite"].median(), 4),
                "CATE_Std"        : round(grp_df["_ite"].std(), 4),
                "Pct_Negatif"     : round((grp_df["_ite"] < 0).mean() * 100, 1),
                "Outcome_Ortalama": round(grp_df[outcome_col].mean(), 2)
                    if outcome_col in grp_df.columns else np.nan,
            })
    segment_df = pd.DataFrame(seg_rows)


    work["_quartile"] = pd.qcut(
        work["_ite"], q=4,
        labels=["Q1_EnDüşük", "Q2_Düşük", "Q3_Yüksek", "Q4_EnYüksek"],
        duplicates="drop",
    )

    profile_rows = []
    numeric_cols = work.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [
        c for c in numeric_cols
        if c not in ["_ite"] + ALL_TREATMENTS + OUTCOMES
    ]

    for q_label, q_df in work.groupby("_quartile"):
        row = {
            "Treatment"        : treatment_col,
            "ITE_Quartile"     : str(q_label),
            "N"                : len(q_df),
            "CATE_Ortalama"    : round(q_df["_ite"].mean(), 4),
            "Outcome_Ortalama" : round(q_df[outcome_col].mean(), 2)
                if outcome_col in q_df.columns else np.nan,
        }

        for cat in ["urun_tipi", "kumas_tipi"]:
            if cat in q_df.columns:
                row[f"En_Sik_{cat}"] = q_df[cat].mode().iloc[0] \
                    if not q_df[cat].mode().empty else "—"

        for col in numeric_cols[:8]:
            row[f"Ort_{col}"] = round(q_df[col].mean(), 3)
        profile_rows.append(row)

    quartile_df = pd.DataFrame(profile_rows)
    return segment_df, quartile_df


print("\n" + "=" * 80)
print("4. ANA CAUSAL FOREST MODELLERİ")
print("=" * 80)

results = {}
for t, config in CONFOUNDER_SETS.items():
    print(f"\n▶ {config['label']}")
    res = run_causal_forest(df, t, config, n_estimators=500, seed=42)
    results[t] = res
    ci_zero  = res["ate_lb"] <= 0 <= res["ate_ub"]
    sig_text = "CI sıfırı içeriyor" if ci_zero else "CI sıfırı içermiyor ✓"
    print(
        f"  ATE = {res['ate']:.4f} [{res['ate_lb']:.4f}, {res['ate_ub']:.4f}] | {sig_text}"
    )


print("\n" + "=" * 80)
print("5. PLACEBO TESTİ")
print("=" * 80)

rng = np.random.default_rng(99)
placebo_results = {}

for t, config in CONFOUNDER_SETS.items():
    Y        = df[config["outcome"]].values
    X        = prepare_X(df, config["confounders"])
    W_placebo = rng.binomial(1, 0.5, size=len(df)).astype(int)

    cf_p = CausalForestDML(
        model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=1, n_jobs=-1),
        model_t=RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=1, n_jobs=-1),
        n_estimators=300, min_samples_leaf=10,
        random_state=1, inference=True, discrete_treatment=True, cv=5,
    )
    cf_p.fit(Y, W_placebo, X=X)

    ate_p    = float(np.atleast_1d(cf_p.ate(X))[0])
    ate_ci_p = cf_p.ate_interval(X, alpha=0.05)
    placebo_results[t] = {
        "ate"   : ate_p,
        "ate_lb": float(np.atleast_1d(ate_ci_p[0])[0]),
        "ate_ub": float(np.atleast_1d(ate_ci_p[1])[0]),
    }
    print(f"  {t:<32} Placebo ATE = {ate_p:.4f}")


print("\n" + "=" * 80)
print("6. DUYARLILIK ANALİZİ — ARACI DEĞİŞKENLER DAHİL")
print("=" * 80)

sensitivity_results = {}

for t, config in CONFOUNDER_SETS.items():
    available_mediators = [m for m in config["mediators"] if m in df.columns]
    if not available_mediators:
        continue

    conf_extended = config["confounders"] + available_mediators
    Y     = df[config["outcome"]].values
    W     = df[t].values.astype(int)
    X_ext = prepare_X(df, conf_extended)

    cf_s = CausalForestDML(
        model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=42, n_jobs=-1),
        model_t=RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42, n_jobs=-1),
        n_estimators=300, min_samples_leaf=10,
        random_state=42, inference=True, discrete_treatment=True, cv=5,
    )
    cf_s.fit(Y, W, X=X_ext)

    ate_s    = float(np.atleast_1d(cf_s.ate(X_ext))[0])
    ate_ci_s = cf_s.ate_interval(X_ext, alpha=0.05)
    sensitivity_results[t] = {
        "ate"   : ate_s,
        "ate_lb": float(np.atleast_1d(ate_ci_s[0])[0]),
        "ate_ub": float(np.atleast_1d(ate_ci_s[1])[0]),
    }

    print(
        f"  {t:<32} Ana ATE = {results[t]['ate']:.4f} | "
        f"Duyarlılık ATE = {ate_s:.4f} | "
        f"Fark = {ate_s - results[t]['ate']:.4f}"
    )


print("\n" + "=" * 80)
print("7. CATE / ITE ÖZETİ")
print("=" * 80)

cate_summaries = {}
for t, config in CONFOUNDER_SETS.items():
    ite = results[t]["ite"]
    cate_summaries[t] = {
        "ITE_mean"    : float(np.mean(ite)),
        "ITE_std"     : float(np.std(ite)),
        "ITE_p10"     : float(np.percentile(ite, 10)),
        "ITE_p25"     : float(np.percentile(ite, 25)),
        "ITE_median"  : float(np.median(ite)),
        "ITE_p75"     : float(np.percentile(ite, 75)),
        "ITE_p90"     : float(np.percentile(ite, 90)),
        "pct_negative": float(np.mean(ite < 0) * 100),
        "pct_positive": float(np.mean(ite > 0) * 100),
    }
    s = cate_summaries[t]
    print(f"\n  {config['label']}")
    print(f"  Ort ITE: {s['ITE_mean']:.4f} | Medyan: {s['ITE_median']:.4f} | Std: {s['ITE_std']:.4f}")
    print(f"  P10–P90: [{s['ITE_p10']:.4f}, {s['ITE_p90']:.4f}]")
    print(f"  Negatif: %{s['pct_negative']:.1f} | Pozitif: %{s['pct_positive']:.1f}")


print("\n" + "=" * 80)
print("8. CATE SEGMENT ANALİZİ")
print("=" * 80)

all_segment_dfs  = []
all_quartile_dfs = []

for t, config in CONFOUNDER_SETS.items():
    print(f"\n▶ {config['label']}")
    ite = results[t]["ite"]

    seg_df, q_df = cate_segment_analysis(df, ite, t, config)
    all_segment_dfs.append(seg_df)
    all_quartile_dfs.append(q_df)


    for var in SEGMENT_VARS:
        sub = seg_df[seg_df["Segment_Degisken"] == var].copy()
        if sub.empty:
            continue
        best  = sub.loc[sub["CATE_Ortalama"].idxmin()]
        worst = sub.loc[sub["CATE_Ortalama"].idxmax()]
        print(
            f"  [{var}] En etkili grup: {best['Segment_Grubu']} "
            f"(CATE={best['CATE_Ortalama']:.3f}, N={best['N']}) | "
            f"En az etkili: {worst['Segment_Grubu']} "
            f"(CATE={worst['CATE_Ortalama']:.3f}, N={worst['N']})"
        )

cate_segment_df  = pd.concat(all_segment_dfs,  ignore_index=True)
cate_quartile_df = pd.concat(all_quartile_dfs, ignore_index=True)


parti_pivot_df = pd.DataFrame()
parti_sub = cate_segment_df[cate_segment_df["Segment_Degisken"] == "parti_buyuklugu"].copy()
if not parti_sub.empty:

    parti_sub["Treatment_Label"] = parti_sub["Treatment"].map(
        lambda t: labels_short.get(t, t).replace("\n", " ")
    )
    parti_pivot_df = parti_sub.pivot_table(
        index="Segment_Grubu",
        columns="Treatment_Label",
        values="CATE_Ortalama",
        aggfunc="mean",
    )

    row_order = [r for r in
                 ["parti_buyuklugu_Düşük", "parti_buyuklugu_Orta", "parti_buyuklugu_Yüksek"]
                 if r in parti_pivot_df.index]
    if row_order:
        parti_pivot_df = parti_pivot_df.reindex(row_order)


karar_rows = []
for t, config in CONFOUNDER_SETS.items():
    r       = results[t]
    ci_zero = r["ate_lb"] <= 0 <= r["ate_ub"]
    sub     = cate_segment_df[cate_segment_df["Treatment"] == t]
    if sub.empty:
        continue

    best_idx  = sub["CATE_Ortalama"].idxmin()
    worst_idx = sub["CATE_Ortalama"].idxmax()
    best      = sub.loc[best_idx]
    worst     = sub.loc[worst_idx]


    if best["CATE_Ortalama"] < 0 and not ci_zero:
        yorum = (
            f"'{best['Segment_Grubu']}' segmentinde güçlü azaltım potansiyeli "
            f"gözlemlenmektedir. Bu segmente öncelikli uygulama önerilir."
        )
    elif best["CATE_Ortalama"] < 0 and ci_zero:
        yorum = (
            f"'{best['Segment_Grubu']}' segmentinde azaltım yönünde bir eğilim "
            f"görülmekle birlikte, ana model güven aralığının sıfırı içermesi "
            f"nedeniyle karar destek açısından dikkatle değerlendirilmelidir."
        )
    else:
        yorum = (
            f"Hiçbir segmentte belirgin azaltım etkisi gözlemlenmemektedir. "
            f"Bu treatment için ek veri veya model gözden geçirmesi önerilir."
        )

    karar_rows.append({
        "Treatment"                       : t,
        "Outcome"                         : config["outcome"],
        "ATE"                             : round(r["ate"], 4),
        "CI_Sifir_Icerir"                 : ci_zero,
        "En_Guclu_Segment_Degisken"       : best["Segment_Degisken"],
        "En_Guclu_Segment_Grubu"          : best["Segment_Grubu"],
        "En_Guclu_CATE"                   : round(best["CATE_Ortalama"], 4),
        "En_Guclu_N"                      : int(best["N"]),
        "En_Zayif_Segment_Degisken"       : worst["Segment_Degisken"],
        "En_Zayif_Segment_Grubu"          : worst["Segment_Grubu"],
        "En_Zayif_CATE"                   : round(worst["CATE_Ortalama"], 4),
        "En_Zayif_N"                      : int(worst["N"]),
        "Karar_Destek_Yorumu"             : yorum,
    })
best_segments_df = pd.DataFrame(karar_rows)


esik_rows = []
for var in ["parti_buyuklugu", "makine_yasi_yil", "kapasite_kullanim_pct"]:
    if var not in df.columns:
        continue
    try:
        _, bins = pd.qcut(df[var].dropna(), q=3, retbins=True, duplicates="drop")
        esik_rows.append({
            "Degisken"   : var,
            "Dusuk_Aralik": f"{bins[0]:.2f} – {bins[1]:.2f}",
            "Orta_Aralik" : f"{bins[1]:.2f} – {bins[2]:.2f}",
            "Yuksek_Aralik": f"{bins[2]:.2f} – {bins[3]:.2f}",
            "Yontem"     : "Üç eşit frekanslı grup (qcut)",
        })
    except Exception as e:
        esik_rows.append({
            "Degisken": var, "Dusuk_Aralik": "Hata",
            "Orta_Aralik": str(e), "Yuksek_Aralik": "", "Yontem": "",
        })
segment_esik_df = pd.DataFrame(esik_rows)

print("\n" + "=" * 80)
print("9b. EK DATAFRAME'LER OLUŞTURULDU")
print("=" * 80)
print(f"  Parti pivot satır sayısı : {len(parti_pivot_df)}")
print(f"  Karar destek satır sayısı: {len(best_segments_df)}")
print(f"  Segment eşik satır sayısı: {len(segment_esik_df)}")
if not segment_esik_df.empty:
    print(segment_esik_df.to_string(index=False))


print("\n" + "=" * 80)
print("9. EXCEL ÇIKTI DOSYASI HAZIRLANIYOR")
print("=" * 80)

excel_path = "/content/outputs/causal_forest_sonuclar.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:


    summary_rows = []
    for t, config in CONFOUNDER_SETS.items():
        r  = results[t]
        pr = placebo_results.get(t, {})
        sr = sensitivity_results.get(t, {})
        s  = cate_summaries[t]
        ci_zero = r["ate_lb"] <= 0 <= r["ate_ub"]
        summary_rows.append({
            "Treatment"               : t,
            "Outcome"                 : config["outcome"],
            "Temel_Confounder_Sayisi" : len(config["base_confounders"]),
            "Toplam_Confounder_Sayisi": len(config["confounders"]),
            "Araci_Degiskenler"       : ", ".join(config["mediators"]),
            "Tedavi_N"                : r["n_treated"],
            "Kontrol_N"               : r["n_control"],
            "ATE"                     : round(r["ate"], 4),
            "CI_Alt_95"               : round(r["ate_lb"], 4),
            "CI_Ust_95"               : round(r["ate_ub"], 4),
            "CI_Sifir_Icerir"         : ci_zero,
            "Etki_Yonu"               : "Azaltici" if r["ate"] < 0 else "Artirici",
            "ITE_Ortalama"            : round(s["ITE_mean"], 4),
            "ITE_Medyan"              : round(s["ITE_median"], 4),
            "ITE_Std"                 : round(s["ITE_std"], 4),
            "ITE_P10"                 : round(s["ITE_p10"], 4),
            "ITE_P90"                 : round(s["ITE_p90"], 4),
            "Pct_Negatif_Etki"        : round(s["pct_negative"], 1),
            "Pct_Pozitif_Etki"        : round(s["pct_positive"], 1),
            "Placebo_ATE"             : round(pr.get("ate", np.nan), 4),
            "Placebo_CI_Alt"          : round(pr.get("ate_lb", np.nan), 4),
            "Placebo_CI_Ust"          : round(pr.get("ate_ub", np.nan), 4),
            "Duyarlilik_ATE"          : round(sr.get("ate", np.nan), 4),
            "Duyarlilik_CI_Alt"       : round(sr.get("ate_lb", np.nan), 4),
            "Duyarlilik_CI_Ust"       : round(sr.get("ate_ub", np.nan), 4),
        })
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Ozet_Sonuclar", index=False)
    print("  ✓ Sayfa 1: Ozet_Sonuclar")


    ite_df = pd.DataFrame({"batch_index": range(len(df))})
    for t in ALL_TREATMENTS:
        ite_df[f"ite_{t}"]    = results[t]["ite"]
        ite_df[f"ite_lb_{t}"] = results[t]["ite_lb"]
        ite_df[f"ite_ub_{t}"] = results[t]["ite_ub"]
    ite_df.to_excel(writer, sheet_name="ITE_Satir_Bazli", index=False)
    print("  ✓ Sayfa 2: ITE_Satir_Bazli")


    mapping_rows = []
    for t, cfg in CONFOUNDER_SETS.items():
        mapping_rows.append({
            "Treatment"                    : t,
            "Outcome"                      : cfg["outcome"],
            "Temel_Confounderlar"          : ", ".join(cfg["base_confounders"]),
            "Diger_Treatmentlar_Confounder": ", ".join(cfg["other_treatments_as_confounders"]),
            "Tum_Confounderlar"            : ", ".join(cfg["confounders"]),
            "Araci_Degiskenler"            : ", ".join(cfg["mediators"]),
        })
    pd.DataFrame(mapping_rows).to_excel(
        writer, sheet_name="Treatment_Outcome_Eslestirme", index=False
    )
    print("  ✓ Sayfa 3: Treatment_Outcome_Eslestirme")


    fi_rows = []
    for t, config in CONFOUNDER_SETS.items():
        for feat, imp in sorted(results[t]["feat_imp"].items(), key=lambda x: -x[1]):
            fi_rows.append({"Treatment": t, "Degisken": feat, "Onem_Skoru": round(imp, 6)})
    pd.DataFrame(fi_rows).to_excel(writer, sheet_name="Feature_Importance", index=False)
    print("  ✓ Sayfa 4: Feature_Importance")


    compare_rows = []
    for t, config in CONFOUNDER_SETS.items():
        r  = results[t]
        pr = placebo_results.get(t, {})
        sr = sensitivity_results.get(t, {})
        compare_rows.append({
            "Treatment"         : t,
            "Ana_ATE"           : round(r["ate"], 4),
            "Ana_CI_Alt"        : round(r["ate_lb"], 4),
            "Ana_CI_Ust"        : round(r["ate_ub"], 4),
            "Placebo_ATE"       : round(pr.get("ate", np.nan), 4),
            "Placebo_CI_Alt"    : round(pr.get("ate_lb", np.nan), 4),
            "Placebo_CI_Ust"    : round(pr.get("ate_ub", np.nan), 4),
            "Duyarlilik_ATE"    : round(sr.get("ate", np.nan), 4),
            "Duyarlilik_CI_Alt" : round(sr.get("ate_lb", np.nan), 4),
            "Duyarlilik_CI_Ust" : round(sr.get("ate_ub", np.nan), 4),
            "Placebo_Fark"      : round(pr.get("ate", np.nan) - r["ate"], 4),
            "Duyarlilik_Fark"   : round(sr.get("ate", np.nan) - r["ate"], 4),
        })
    pd.DataFrame(compare_rows).to_excel(writer, sheet_name="Placebo_Duyarlilik", index=False)
    print("  ✓ Sayfa 5: Placebo_Duyarlilik")


    cate_segment_df.to_excel(writer, sheet_name="CATE_Segment_Analizi", index=False)
    print("  ✓ Sayfa 6: CATE_Segment_Analizi  [YENİ]")
    print("            (urun_tipi / kumas_tipi / kapasite / makine_yasi / parti_buyuklugu bazında)")


    cate_quartile_df.to_excel(writer, sheet_name="CATE_Quartile_Profil", index=False)
    print("  ✓ Sayfa 7: CATE_Quartile_Profil  [YENİ]")
    print("            (Q1=en düşük etki ... Q4=en yüksek etki)")


    if "urun_tipi" in df.columns:
        pivot_rows = []
        for t in ALL_TREATMENTS:
            work = df[["urun_tipi"]].copy()
            work["_ite"] = results[t]["ite"]
            grp = work.groupby("urun_tipi")["_ite"].mean()
            for urun, val in grp.items():
                pivot_rows.append({"Treatment": t, "Urun_Tipi": urun, "CATE_Ortalama": round(val, 4)})
        pivot_df = pd.DataFrame(pivot_rows)
        pivot_table = pivot_df.pivot(index="Urun_Tipi", columns="Treatment", values="CATE_Ortalama")
        pivot_table.to_excel(writer, sheet_name="CATE_Pivot_UrunxTreatment")
        print("  ✓ Sayfa 8: CATE_Pivot_UrunxTreatment  [YENİ]")
        print("            (urun_tipi × treatment ısı haritası pivot tablosu)")


    if not parti_pivot_df.empty:
        parti_pivot_df.to_excel(writer, sheet_name="CATE_Pivot_PartixTreatment")
        print("  ✓ Sayfa 9:  CATE_Pivot_PartixTreatment  [YENİ v4]")
        print("             (parti_buyuklugu grubu × treatment ısı haritası pivot tablosu)")
    else:
        print("  ⚠ Sayfa 9:  CATE_Pivot_PartixTreatment atlandı (parti_buyuklugu segmenti bulunamadı)")


    if not best_segments_df.empty:
        best_segments_df.to_excel(writer, sheet_name="CATE_Karar_Destek_Ozeti", index=False)
        print("  ✓ Sayfa 10: CATE_Karar_Destek_Ozeti  [YENİ v4]")
        print("             (treatment bazlı en güçlü/en zayıf segment ve karar destek yorumu)")
    else:
        print("  ⚠ Sayfa 10: CATE_Karar_Destek_Ozeti atlandı (veri yok)")


    if not segment_esik_df.empty:
        segment_esik_df.to_excel(writer, sheet_name="Segment_Esik_Degerleri", index=False)
        print("  ✓ Sayfa 11: Segment_Esik_Degerleri  [YENİ v4]")
        print("             (parti_buyuklugu / makine_yasi_yil / kapasite_kullanim_pct grup sınırları)")
    else:
        print("  ⚠ Sayfa 11: Segment_Esik_Degerleri atlandı (veri yok)")

print(f"\n  Excel dosyası kaydedildi: {excel_path}  (11 sayfa)")


print("\n" + "=" * 80)
print("10. GÖRSEL ÇIKTILAR")
print("=" * 80)

plt.rcParams.update({
    "figure.dpi"      : 150,
    "font.family"     : "DejaVu Sans",
    "axes.spines.top" : False,
    "axes.spines.right": False,
})


fig1, ax = plt.subplots(figsize=(12, 7))
fig1.patch.set_facecolor(COLORS["bg"])
ax.set_facecolor(COLORS["bg"])

for i, t in enumerate(ALL_TREATMENTS):
    r     = results[t]
    color = COLORS["primary"] if not (r["ate_lb"] <= 0 <= r["ate_ub"]) else COLORS["neutral"]
    ax.plot([r["ate_lb"], r["ate_ub"]], [i, i], lw=3, color=color, alpha=0.7)
    ax.scatter(r["ate"], i, s=120, color=color, zorder=5)
    pr = placebo_results.get(t, {})
    if pr:
        ax.scatter(pr["ate"], i + 0.3, s=60, color=COLORS["placebo"], marker="D", zorder=4, alpha=0.8)
        ax.plot([pr["ate_lb"], pr["ate_ub"]], [i + 0.3, i + 0.3], lw=1.5, color=COLORS["placebo"], alpha=0.5)

ax.axvline(0, color="black", lw=1.2, ls="--", alpha=0.6)
ax.set_yticks([i + 0.15 for i in range(len(ALL_TREATMENTS))])
ax.set_yticklabels(
    [f"{labels_short[t]}\n→ {CONFOUNDER_SETS[t]['outcome']}" for t in ALL_TREATMENTS],
    fontsize=9,
)
ax.set_xlabel("Ortalama Tedavi Etkisi (ATE) — kgCO₂e", fontsize=11)
ax.set_title(
    "Causal Forest — ATE Forest Plot\n(Gri baklava: Placebo testi)",
    fontsize=12, fontweight="bold", pad=15,
)
handles = [
    mpatches.Patch(color=COLORS["primary"], label="CI sıfırı içermiyor"),
    mpatches.Patch(color=COLORS["neutral"], label="CI sıfırı içeriyor"),
    plt.Line2D([0], [0], marker="D", color="w", markerfacecolor=COLORS["placebo"],
               markersize=8, label="Placebo ATE"),
]
ax.legend(handles=handles, fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig("/content/outputs/fig1_ate_forest_plot.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig1_ate_forest_plot.png")


fig2, axes = plt.subplots(2, 3, figsize=(15, 9))
fig2.patch.set_facecolor(COLORS["bg"])
fig2.suptitle("Causal Forest — ITE Bireysel Tedavi Etkisi Dağılımları",
              fontsize=14, fontweight="bold", y=1.01)
for ax_sub, t in zip(axes.flatten(), ALL_TREATMENTS):
    ax_sub.set_facecolor(COLORS["bg"])
    ite = results[t]["ite"]
    ax_sub.hist(ite, bins=40, color=COLORS["primary"], alpha=0.75, edgecolor="white")
    ax_sub.axvline(0, color="black", ls="--", lw=1.2)
    ax_sub.axvline(np.mean(ite), color=COLORS["secondary"], ls="-", lw=2,
                   label=f"Ort={np.mean(ite):.2f}")
    ax_sub.set_title(labels_short[t], fontsize=10, fontweight="bold")
    ax_sub.set_xlabel("ITE — kgCO₂e", fontsize=9)
    ax_sub.set_ylabel("Frekans", fontsize=9)
    ax_sub.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/content/outputs/fig2_ite_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig2_ite_distributions.png")


fig3, axes = plt.subplots(2, 3, figsize=(16, 10))
fig3.patch.set_facecolor(COLORS["bg"])
fig3.suptitle("Causal Forest — Heterojenite Sürücüleri", fontsize=14, fontweight="bold")
for ax_sub, t in zip(axes.flatten(), ALL_TREATMENTS):
    ax_sub.set_facecolor(COLORS["bg"])
    fi        = results[t]["feat_imp"]
    fi_sorted = sorted(fi.items(), key=lambda x: x[1], reverse=True)[:10]
    names     = [x[0] for x in fi_sorted]
    vals      = [x[1] for x in fi_sorted]
    names_s   = [
        n.replace("urun_tipi_", "ürün:")
         .replace("kumas_tipi_", "kumaş:")
         .replace("tasima_modu_", "taşıma:")
         .replace("ay_", "ay:").replace("t_", "T:").replace("_", " ")
        for n in names
    ]
    colors_b = [COLORS["primary"] if v > np.mean(vals) else COLORS["neutral"] for v in vals]
    ax_sub.barh(names_s[::-1], vals[::-1], color=colors_b[::-1], edgecolor="white")
    ax_sub.set_title(labels_short[t], fontsize=10, fontweight="bold")
    ax_sub.set_xlabel("Önem Skoru", fontsize=8)
plt.tight_layout()
plt.savefig("/content/outputs/fig3_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig3_feature_importance.png")


fig4, ax = plt.subplots(figsize=(12, 6))
fig4.patch.set_facecolor(COLORS["bg"])
ax.set_facecolor(COLORS["bg"])
x   = np.arange(len(ALL_TREATMENTS))
w   = 0.25
ax.bar(x - w, [results[t]["ate"]                              for t in ALL_TREATMENTS],
       w, label="Ana Model ATE",  color=COLORS["primary"],     alpha=0.85)
ax.bar(x,     [placebo_results[t]["ate"]                      for t in ALL_TREATMENTS],
       w, label="Placebo ATE",    color=COLORS["placebo"],     alpha=0.85)
ax.bar(x + w, [sensitivity_results.get(t, {}).get("ate", np.nan) for t in ALL_TREATMENTS],
       w, label="Duyarlılık ATE", color=COLORS["sensitivity"], alpha=0.85)
ax.axhline(0, color="black", lw=1.2, ls="--")
ax.set_xticks(x)
ax.set_xticklabels([labels_short[t] for t in ALL_TREATMENTS], fontsize=9)
ax.set_ylabel("ATE — kgCO₂e", fontsize=11)
ax.set_title("Ana Model | Placebo | Duyarlılık — ATE Karşılaştırması",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("/content/outputs/fig4_ate_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig4_ate_comparison.png")


fig5, ax = plt.subplots(figsize=(12, 6))
fig5.patch.set_facecolor(COLORS["bg"])
ax.set_facecolor(COLORS["bg"])
bp = ax.boxplot([results[t]["ite"] for t in ALL_TREATMENTS],
                patch_artist=True, notch=False, medianprops=dict(color="black", lw=2))
for patch in bp["boxes"]:
    patch.set_facecolor(COLORS["primary"])
    patch.set_alpha(0.6)
ax.axhline(0, color="black", ls="--", lw=1.2)
ax.set_xticks(range(1, len(ALL_TREATMENTS) + 1))
ax.set_xticklabels([labels_short[t] for t in ALL_TREATMENTS], fontsize=9)
ax.set_ylabel("ITE — kgCO₂e", fontsize=11)
ax.set_title("CATE Dağılımı — Bireysel Tedavi Etkisi Kutu Grafikleri",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/outputs/fig5_cate_boxplots.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig5_cate_boxplots.png")


fig6, axes6 = plt.subplots(2, 3, figsize=(17, 11))
fig6.patch.set_facecolor(COLORS["bg"])
fig6.suptitle(
    "CATE Segment Analizi — Ürün Tipi ve Kumaş Tipi Bazında Ortalama Tedavi Etkisi",
    fontsize=13, fontweight="bold",
)

for ax_sub, t in zip(axes6.flatten(), ALL_TREATMENTS):
    ax_sub.set_facecolor(COLORS["bg"])
    sub = cate_segment_df[
        (cate_segment_df["Treatment"] == t) &
        (cate_segment_df["Segment_Degisken"].isin(["urun_tipi", "kumas_tipi"]))
    ].copy()

    if sub.empty:
        ax_sub.set_visible(False)
        continue


    sub["label"] = sub["Segment_Degisken"].str[:3] + ":" + sub["Segment_Grubu"].str[:12]
    sub = sub.sort_values("CATE_Ortalama")

    bar_colors = [COLORS["high_effect"] if v < 0 else COLORS["low_effect"]
                  for v in sub["CATE_Ortalama"]]
    bars = ax_sub.barh(sub["label"], sub["CATE_Ortalama"],
                       color=bar_colors, edgecolor="white", alpha=0.85)
    ax_sub.axvline(0, color="black", lw=1.0, ls="--", alpha=0.7)


    for bar, (_, row) in zip(bars, sub.iterrows()):
        x_pos = bar.get_width()
        offset = 0.001 if x_pos >= 0 else -0.001
        ax_sub.text(
            x_pos + offset, bar.get_y() + bar.get_height() / 2,
            f"N={row['N']}", va="center",
            ha="left" if x_pos >= 0 else "right",
            fontsize=7, color="#374151",
        )

    ax_sub.set_title(labels_short[t], fontsize=10, fontweight="bold")
    ax_sub.set_xlabel("Ort. CATE — kgCO₂e", fontsize=8)

handles6 = [
    mpatches.Patch(color=COLORS["high_effect"], label="Negatif CATE (emisyon azaltıcı)"),
    mpatches.Patch(color=COLORS["low_effect"],  label="Pozitif CATE (emisyon artırıcı)"),
]
fig6.legend(handles=handles6, loc="lower center", ncol=2, fontsize=9, frameon=False)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("/content/outputs/fig6_cate_segment_urun_kumas.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig6_cate_segment_urun_kumas.png  [YENİ]")


fig7, axes7 = plt.subplots(2, 3, figsize=(17, 11))
fig7.patch.set_facecolor(COLORS["bg"])
fig7.suptitle(
    "CATE Segment Analizi — Kapasite / Makine Yaşı / Parti Büyüklüğü Bazında Ortalama Tedavi Etkisi",
    fontsize=13, fontweight="bold",
)

for ax_sub, t in zip(axes7.flatten(), ALL_TREATMENTS):
    ax_sub.set_facecolor(COLORS["bg"])
    sub = cate_segment_df[
        (cate_segment_df["Treatment"] == t) &
        (cate_segment_df["Segment_Degisken"].isin(
            ["kapasite_kullanim_pct", "makine_yasi_yil", "parti_buyuklugu"]
        ))
    ].copy()

    if sub.empty:
        ax_sub.set_visible(False)
        continue

    sub["label"] = (
        sub["Segment_Degisken"]
            .str.replace("kapasite_kullanim_pct", "Kapasite", regex=False)
            .str.replace("makine_yasi_yil",        "Mak.Yaşı", regex=False)
            .str.replace("parti_buyuklugu",         "Parti",    regex=False)
        + ":" + sub["Segment_Grubu"].str.split("_").str[-1]
    )
    sub = sub.sort_values("CATE_Ortalama")
    bar_colors = [COLORS["high_effect"] if v < 0 else COLORS["low_effect"]
                  for v in sub["CATE_Ortalama"]]
    bars = ax_sub.barh(sub["label"], sub["CATE_Ortalama"],
                       color=bar_colors, edgecolor="white", alpha=0.85)
    ax_sub.axvline(0, color="black", lw=1.0, ls="--", alpha=0.7)

    for bar, (_, row) in zip(bars, sub.iterrows()):
        x_pos = bar.get_width()
        offset = 0.001 if x_pos >= 0 else -0.001
        ax_sub.text(
            x_pos + offset, bar.get_y() + bar.get_height() / 2,
            f"N={row['N']}", va="center",
            ha="left" if x_pos >= 0 else "right",
            fontsize=7, color="#374151",
        )

    ax_sub.set_title(labels_short[t], fontsize=10, fontweight="bold")
    ax_sub.set_xlabel("Ort. CATE — kgCO₂e", fontsize=8)

fig7.legend(handles=handles6, loc="lower center", ncol=2, fontsize=9, frameon=False)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("/content/outputs/fig7_cate_segment_kapasite_makine.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ fig7_cate_segment_kapasite_makine.png  [YENİ]")


if "urun_tipi" in df.columns:
    pivot_data = {}
    for t in ALL_TREATMENTS:
        work = df[["urun_tipi"]].copy()
        work["_ite"] = results[t]["ite"]
        pivot_data[labels_short[t].replace("\n", " ")] = \
            work.groupby("urun_tipi")["_ite"].mean()

    heatmap_df = pd.DataFrame(pivot_data)

    fig8, ax8 = plt.subplots(figsize=(14, max(5, len(heatmap_df) * 0.7 + 2)))
    fig8.patch.set_facecolor(COLORS["bg"])
    ax8.set_facecolor(COLORS["bg"])

    vmax = max(abs(heatmap_df.values.max()), abs(heatmap_df.values.min()))
    cmap = plt.cm.RdYlGn_r

    im = ax8.imshow(heatmap_df.values, cmap=cmap, aspect="auto",
                    vmin=-vmax, vmax=vmax)

    ax8.set_xticks(range(len(heatmap_df.columns)))
    ax8.set_xticklabels(heatmap_df.columns, rotation=25, ha="right", fontsize=9)
    ax8.set_yticks(range(len(heatmap_df.index)))
    ax8.set_yticklabels(heatmap_df.index, fontsize=9)


    for i in range(len(heatmap_df.index)):
        for j in range(len(heatmap_df.columns)):
            val = heatmap_df.values[i, j]
            text_color = "white" if abs(val) > vmax * 0.6 else "black"
            ax8.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=8, color=text_color, fontweight="bold")

    plt.colorbar(im, ax=ax8, label="Ortalama CATE (kgCO₂e)", shrink=0.8)
    ax8.set_title(
        "CATE Isı Haritası — Ürün Tipi × Treatment\n"
        "Yeşil: Emisyon azaltıcı etki  |  Kırmızı: Emisyon artırıcı etki",
        fontsize=12, fontweight="bold", pad=12,
    )
    plt.tight_layout()
    plt.savefig("/content/outputs/fig8_cate_heatmap_urun_treatment.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✓ fig8_cate_heatmap_urun_treatment.png  [YENİ]")


profile_vars = ["kapasite_kullanim_pct", "makine_yasi_yil", "parti_buyuklugu"]
profile_vars_present = [v for v in profile_vars if f"Ort_{v}" in cate_quartile_df.columns]

if profile_vars_present:
    fig9, axes9 = plt.subplots(2, 3, figsize=(17, 11))
    fig9.patch.set_facecolor(COLORS["bg"])
    fig9.suptitle(
        "CATE Quartile Profil — Q1 (En Düşük Etki) vs Q4 (En Yüksek Etki) Karşılaştırması",
        fontsize=13, fontweight="bold",
    )

    for ax_sub, t in zip(axes9.flatten(), ALL_TREATMENTS):
        ax_sub.set_facecolor(COLORS["bg"])
        sub = cate_quartile_df[cate_quartile_df["Treatment"] == t].copy()


        sub_q = sub[sub["ITE_Quartile"].isin(["Q1_EnDüşük", "Q4_EnYüksek"])].copy()
        if sub_q.empty or not profile_vars_present:
            ax_sub.set_visible(False)
            continue

        x_pos = np.arange(len(profile_vars_present))
        width = 0.35
        q1_vals = []
        q4_vals = []

        for pv in profile_vars_present:
            col = f"Ort_{pv}"
            q1_row = sub_q[sub_q["ITE_Quartile"] == "Q1_EnDüşük"]
            q4_row = sub_q[sub_q["ITE_Quartile"] == "Q4_EnYüksek"]
            q1_vals.append(q1_row[col].values[0] if not q1_row.empty else np.nan)
            q4_vals.append(q4_row[col].values[0] if not q4_row.empty else np.nan)

        ax_sub.bar(x_pos - width/2, q1_vals, width,
                   label="Q1 (En düşük etki)", color=COLORS["low_effect"], alpha=0.8)
        ax_sub.bar(x_pos + width/2, q4_vals, width,
                   label="Q4 (En yüksek etki)", color=COLORS["high_effect"], alpha=0.8)

        labels_pv = [v.replace("kapasite_kullanim_pct", "Kapasite %")
                      .replace("makine_yasi_yil", "Makine Yaşı")
                      .replace("parti_buyuklugu", "Parti Büyüklüğü")
                     for v in profile_vars_present]
        ax_sub.set_xticks(x_pos)
        ax_sub.set_xticklabels(labels_pv, fontsize=8)
        ax_sub.set_title(labels_short[t], fontsize=10, fontweight="bold")
        ax_sub.set_ylabel("Ortalama Değer", fontsize=8)
        ax_sub.legend(fontsize=7)

    plt.tight_layout()
    plt.savefig("/content/outputs/fig9_cate_quartile_profil.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✓ fig9_cate_quartile_profil.png  [YENİ]")


if not parti_pivot_df.empty:
    fig10, ax10 = plt.subplots(figsize=(14, 4))
    fig10.patch.set_facecolor(COLORS["bg"])
    ax10.set_facecolor(COLORS["bg"])

    hm_vals = parti_pivot_df.values.astype(float)
    vmax10  = np.nanmax(np.abs(hm_vals))
    if vmax10 == 0:
        vmax10 = 1.0

    im10 = ax10.imshow(hm_vals, cmap=plt.cm.RdYlGn_r, aspect="auto",
                       vmin=-vmax10, vmax=vmax10)

    ax10.set_xticks(range(len(parti_pivot_df.columns)))
    ax10.set_xticklabels(parti_pivot_df.columns, rotation=25, ha="right", fontsize=9)
    ax10.set_yticks(range(len(parti_pivot_df.index)))

    row_labels = [
        r.replace("parti_buyuklugu_", "Parti: ")
         .replace("parti_buyuklugu", "Parti")
        for r in parti_pivot_df.index.astype(str)
    ]
    ax10.set_yticklabels(row_labels, fontsize=10)


    for i in range(len(parti_pivot_df.index)):
        for j in range(len(parti_pivot_df.columns)):
            val = hm_vals[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if abs(val) > vmax10 * 0.6 else "black"
            ax10.text(j, i, f"{val:.2f}", ha="center", va="center",
                      fontsize=9, color=text_color, fontweight="bold")

    plt.colorbar(im10, ax=ax10, label="Ortalama CATE (kgCO₂e)", shrink=0.8)
    ax10.set_title(
        "CATE Isı Haritası — Parti Büyüklüğü Grubu × Treatment\n"
        "Yeşil: Emisyon azaltıcı etki  |  Kırmızı: Emisyon artırıcı etki",
        fontsize=12, fontweight="bold", pad=12,
    )
    plt.tight_layout()
    plt.savefig("/content/outputs/fig10_cate_heatmap_parti_treatment.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✓ fig10_cate_heatmap_parti_treatment.png  [YENİ v4]")
else:
    print("  ⚠ fig10 atlandı — parti_buyuklugu segment verisi bulunamadı.")


print("\n" + "=" * 80)
print("11. RAPOR İÇİN YORUM ŞABLONLARI")
print("=" * 80)

for t, config in CONFOUNDER_SETS.items():
    r         = results[t]
    ci_zero   = r["ate_lb"] <= 0 <= r["ate_ub"]
    direction = "azaltıcı" if r["ate"] < 0 else "artırıcı"
    s         = cate_summaries[t]

    seg_sub = cate_segment_df[
        (cate_segment_df["Treatment"] == t) &
        (cate_segment_df["Segment_Degisken"] == "urun_tipi")
    ]
    if not seg_sub.empty:
        best_seg  = seg_sub.loc[seg_sub["CATE_Ortalama"].idxmin()]
        worst_seg = seg_sub.loc[seg_sub["CATE_Ortalama"].idxmax()]
        segment_sentence = (
            f"Segment analizi, bu etkinin homojen olmadığını ortaya koymaktadır: "
            f"'{best_seg['Segment_Grubu']}' ürün tipinde ortalama CATE "
            f"{best_seg['CATE_Ortalama']:.3f} kgCO₂e ile en güçlü azaltıcı etkiye "
            f"sahipken, '{worst_seg['Segment_Grubu']}' ürün tipinde "
            f"{worst_seg['CATE_Ortalama']:.3f} kgCO₂e ile en sınırlı etki gözlemlenmektedir. "
            f"Bu farklılık, müdahale önceliklerinin ürün tipine göre belirlenmesi "
            f"gerektiğine işaret etmektedir."
        )
    else:
        segment_sentence = ""

    sig_text = (
        "Güven aralığının sıfırı içermemesi bulgının istatistiksel olarak daha güçlü "
        "olduğuna işaret etmektedir."
        if not ci_zero else
        "Güven aralığının sıfırı içermesi nedeniyle sonuç tek başına güçlü bir "
        "nedensel etki kanıtı olarak yorumlanmamalıdır."
    )

    print(f"\n--- {config['label']} ---")
    print(
        f"ATE = {r['ate']:.4f} kgCO₂e [{r['ate_lb']:.4f}, {r['ate_ub']:.4f}]\n"
        f"Eş zamanlı müdahaleler ve confounderlar kontrol edildiğinde {t}, "
        f"{config['outcome']} üzerinde {direction} yönde bir etki tahmin edilmektedir. "
        f"{sig_text} "
        f"Gözlemlerin %{s['pct_negative']:.1f}'inde negatif (emisyon azaltıcı) bireysel "
        f"etki tahmin edilmiş olup ITE dağılımının genişliği (std={s['ITE_std']:.3f}) "
        f"heterojen bir etki yapısına işaret etmektedir. "
        f"{segment_sentence}"
    )

print("\n" + "=" * 80)
print("NEDENSELLİK ANALİZİ TAMAMLANDI")
print("=" * 80)
print("\nKaydedilen dosyalar:")
print("  /content/outputs/causal_forest_sonuclar.xlsx              (11 sayfa)")
print("    Sayfa  1: Ozet_Sonuclar")
print("    Sayfa  2: ITE_Satir_Bazli")
print("    Sayfa  3: Treatment_Outcome_Eslestirme")
print("    Sayfa  4: Feature_Importance")
print("    Sayfa  5: Placebo_Duyarlilik")
print("    Sayfa  6: CATE_Segment_Analizi")
print("    Sayfa  7: CATE_Quartile_Profil")
print("    Sayfa  8: CATE_Pivot_UrunxTreatment")
print("    Sayfa  9: CATE_Pivot_PartixTreatment          [YENİ v4]")
print("    Sayfa 10: CATE_Karar_Destek_Ozeti             [YENİ v4]")
print("    Sayfa 11: Segment_Esik_Degerleri              [YENİ v4]")
print()
print("  /content/outputs/fig1_ate_forest_plot.png")
print("  /content/outputs/fig2_ite_distributions.png")
print("  /content/outputs/fig3_feature_importance.png")
print("  /content/outputs/fig4_ate_comparison.png")
print("  /content/outputs/fig5_cate_boxplots.png")
print("  /content/outputs/fig6_cate_segment_urun_kumas.png")
print("  /content/outputs/fig7_cate_segment_kapasite_makine.png")
print("  /content/outputs/fig8_cate_heatmap_urun_treatment.png")
print("  /content/outputs/fig9_cate_quartile_profil.png")
print("  /content/outputs/fig10_cate_heatmap_parti_treatment.png    [YENİ v4]")

print("\n\n" + "=" * 80)
print("13. KARŞIOLGUSAL ANALİZ — ÜRETİM PARTİSİ BAZINDA MÜDAHALE ETKİSİ")
print("=" * 80)


_csv_path = "/content/tekstil_karbon_datasetM.csv"

try:
    _raw_id_df = pd.read_csv(_csv_path, usecols=lambda c: c == "batch_id")
    if "batch_id" in _raw_id_df.columns and len(_raw_id_df) == len(df):
        _has_batch_id = True
        print("  batch_id başarıyla yüklendi.")
    else:
        _has_batch_id = False
        print("  ⚠ batch_id sütunu bulunamadı veya satır sayısı eşleşmiyor; "
              "batch_index kullanılacak.")
except Exception as _e:
    _has_batch_id = False
    print(f"  ⚠ batch_id yüklenemedi ({_e}); batch_index kullanılacak.")


_cf_base = df.copy()
_cf_base["batch_index"] = np.arange(len(df))
if _has_batch_id:
    _cf_base["batch_id"] = _raw_id_df["batch_id"].values
else:
    _cf_base["batch_id"] = _cf_base["batch_index"].apply(lambda x: f"IDX_{x}")


COUNTERFACTUAL_TREATMENTS = [
    "t_yenilenebilir_enerji",
    "t_geri_donusum_prog",
]


PROFILE_VARS = {
    "t_yenilenebilir_enerji": [
        "urun_tipi", "kumas_tipi", "parti_buyuklugu", "kumas_gramaj_g_m2",
        "uretim_suresi_dk_adet", "vardiya_sayisi", "kapasite_kullanim_pct",
        "makine_yasi_yil", "ay", "elektrik_kwh",
    ],
    "t_geri_donusum_prog": [
        "urun_tipi", "kumas_tipi", "parti_buyuklugu", "hammadde_ton",
        "kumas_gramaj_g_m2", "kapasite_kullanim_pct", "ay", "tekstil_atik_kg",
    ],
}


def _scope_label(outcome_col):
    mapping = {
        "scope2_kgCO2e"           : "Scope 2",
        "scope3_tasima_kgCO2e"    : "Scope 3 — Taşıma",
        "scope3_atik_kgCO2e"      : "Scope 3 — Atık",
        "scope3_kimyasal_kgCO2e"  : "Scope 3 — Kimyasal",
    }
    return mapping.get(outcome_col, outcome_col)


all_parti_rows   = []
all_ozet_rows    = []
all_top10_frames = []

for _t in COUNTERFACTUAL_TREATMENTS:

    _outcome_col = CONFOUNDER_SETS[_t]["outcome"]
    _scope       = _scope_label(_outcome_col)
    _label       = CONFOUNDER_SETS[_t]["label"]

    print(f"\n▶ {_label}")
    print(f"  Outcome : {_outcome_col}  ({_scope})")


    _mask = (_cf_base[_t] == 0).values

    _n_untreated = int(_mask.sum())
    print(f"  Treatment uygulanmamış parti sayısı: {_n_untreated}")

    if _n_untreated == 0:
        print("  ⚠ Bu treatment için treatment=0 gözlem bulunamadı, atlandı.")
        continue


    _obs    = _cf_base.loc[_mask, _outcome_col].values.astype(float)
    _ite    = results[_t]["ite"][_mask].astype(float)
    _ite_lb = results[_t]["ite_lb"][_mask].astype(float)
    _ite_ub = results[_t]["ite_ub"][_mask].astype(float)


    _cf_outcome_raw = _obs + _ite


    _cf_outcome = np.maximum(_cf_outcome_raw, 0.0)


    _reduction_amt = _obs - _cf_outcome


    with np.errstate(divide="ignore", invalid="ignore"):
        _reduction_pct = np.where(
            _obs != 0,
            _reduction_amt / _obs * 100,
            np.nan,
        )


    _direction = np.where(_ite < 0, "Azaltıcı",
                 np.where(_ite > 0, "Artırıcı", "Nötr"))


    _profile_cols = [c for c in PROFILE_VARS.get(_t, []) if c in _cf_base.columns]


    _parti_df = pd.DataFrame({
        "batch_id"              : _cf_base.loc[_mask, "batch_id"].values,
        "batch_index"           : _cf_base.loc[_mask, "batch_index"].values,
        "treatment_variable"    : _t,
        "outcome_variable"      : _outcome_col,
        "outcome_scope"         : _scope,
        "treatment_status"      : 0,
        "observed_outcome"      : np.round(_obs, 4),
        "ITE"                   : np.round(_ite, 4),
        "ITE_CI_alt"            : np.round(_ite_lb, 4),
        "ITE_CI_ust"            : np.round(_ite_ub, 4),
        "counterfactual_outcome": np.round(_cf_outcome, 4),
        "reduction_amount"      : np.round(_reduction_amt, 4),
        "reduction_percent"     : np.round(_reduction_pct, 2),
        "effect_direction"      : _direction,
    })


    for _col in _profile_cols:
        _parti_df[_col] = _cf_base.loc[_mask, _col].values

    all_parti_rows.append(_parti_df)


    _positive_reduction_mask = _reduction_amt > 0
    _n_positive  = int(_positive_reduction_mask.sum())
    _n_non_reduct = _n_untreated - _n_positive

    _ozet = {
        "treatment_variable"              : _t,
        "outcome_variable"                : _outcome_col,
        "outcome_scope"                   : _scope,
        "untreated_batch_count"           : _n_untreated,
        "mean_observed_outcome"           : round(float(np.mean(_obs)), 4),
        "mean_ITE"                        : round(float(np.mean(_ite)), 4),
        "mean_counterfactual_outcome"     : round(float(np.mean(_cf_outcome)), 4),
        "mean_reduction_amount"           : round(float(np.mean(_reduction_amt)), 4),
        "mean_reduction_percent"          : round(float(np.nanmean(_reduction_pct)), 2),
        "total_reduction_potential"       : round(float(np.sum(_reduction_amt[_positive_reduction_mask])), 4),
        "positive_reduction_batch_count"  : _n_positive,
        "non_reduction_or_increase_batch_count": _n_non_reduct,
        "min_reduction_amount"            : round(float(np.min(_reduction_amt)), 4),
        "max_reduction_amount"            : round(float(np.max(_reduction_amt)), 4),
        "median_reduction_amount"         : round(float(np.median(_reduction_amt)), 4),
    }
    all_ozet_rows.append(_ozet)


    _top10_df = _parti_df[_parti_df["effect_direction"] == "Azaltıcı"].copy()
    _top10_df = _top10_df.sort_values("reduction_amount", ascending=False).head(10).copy()
    _top10_df.insert(0, "rank", range(1, len(_top10_df) + 1))
    all_top10_frames.append(_top10_df)


    print(f"\n  [KARŞIOLGUSAL ÖZET — {_label}]")
    print(f"  Treatment uygulanmamış parti sayısı      : {_n_untreated}")
    print(f"  Ortalama gözlemlenen {_outcome_col:<28}: {np.mean(_obs):.2f} kgCO₂e")
    print(f"  Ortalama karşıolgusal outcome            : {np.mean(_cf_outcome):.2f} kgCO₂e")
    print(f"  Ortalama azaltım miktarı (ITE ort.)      : {np.mean(_reduction_amt):.2f} kgCO₂e")
    print(f"  Ortalama yüzdesel azaltım                : %{np.nanmean(_reduction_pct):.2f}")
    print(f"  Toplam azaltım potansiyeli (azaltıcı pt.): {np.sum(_reduction_amt[_positive_reduction_mask]):.2f} kgCO₂e")
    print(f"  Azaltım yönünde parti sayısı             : {_n_positive} / {_n_untreated}")
    print(f"\n  Model sonuçlarına göre bu partilerde {_t} uygulanmış olsaydı,")
    print(f"  {_outcome_col} üzerinde azaltıcı yönde bir potansiyele işaret")
    print(f"  etmektedir. Üretim partisi bazında farklılaşan bir etki yapısı")
    print(f"  görülmektedir; karşıolgusal outcome değerleri tahmindir.")


    print(f"\n  En yüksek azaltım potansiyeline sahip ilk 3 parti:")
    for _, _row in _top10_df.head(3).iterrows():
        print(
            f"    Rank {int(_row['rank'])} | batch_id={_row['batch_id']} | "
            f"Gözlem={_row['observed_outcome']:.2f} | "
            f"Karşıolgusal={_row['counterfactual_outcome']:.2f} | "
            f"Azaltım={_row['reduction_amount']:.2f} kgCO₂e "
            f"(%{_row['reduction_percent']:.1f})"
        )


if all_parti_rows:
    _full_parti_df = pd.concat(all_parti_rows, ignore_index=True)
else:
    _full_parti_df = pd.DataFrame()

if all_ozet_rows:
    _ozet_df = pd.DataFrame(all_ozet_rows)
else:
    _ozet_df = pd.DataFrame()

if all_top10_frames:
    _top10_full_df = pd.concat(all_top10_frames, ignore_index=True)
else:
    _top10_full_df = pd.DataFrame()


_karsio_excel_path = "/content/outputs/karsiolgusal_sonuclar.xlsx"

print("\n" + "-" * 60)
print("Karşıolgusal Excel dosyası yazılıyor...")

with pd.ExcelWriter(_karsio_excel_path, engine="openpyxl") as _writer:


    if not _ozet_df.empty:
        _ozet_df.to_excel(_writer, sheet_name="Karsio_Ozet", index=False)
        print("  ✓ Sayfa: Karsio_Ozet")


    if not _full_parti_df.empty:
        _full_parti_df.to_excel(_writer, sheet_name="Karsio_Parti_Bazli", index=False)
        print(f"  ✓ Sayfa: Karsio_Parti_Bazli  ({len(_full_parti_df)} satır)")


    if not _top10_full_df.empty:
        _top10_full_df.to_excel(_writer, sheet_name="Karsio_Top10", index=False)
        print(f"  ✓ Sayfa: Karsio_Top10  ({len(_top10_full_df)} satır)")

print(f"\n  Karşıolgusal Excel kaydedildi: {_karsio_excel_path}")


print("\n" + "-" * 60)
print("Karşıolgusal analiz görselleri oluşturuluyor...")


KARSIO_LABELS = {
    "t_yenilenebilir_enerji": "Yenilenebilir Enerji\n(Scope 2)",
    "t_geri_donusum_prog"   : "Geri Dönüşüm Programı\n(Scope 3 Atık)",
}

KARSIO_PROFILE_CONFIG = {
    "t_yenilenebilir_enerji": {
        "x_col": "elektrik_kwh",
        "x_label": "Elektrik tüketimi (kWh)",
        "title": "Yenilenebilir enerji: Elektrik tüketimi ve azaltım ilişkisi",
        "profile_cols": [
            "parti_buyuklugu",
            "elektrik_kwh",
            "kapasite_kullanim_pct",
            "makine_yasi_yil",
        ],
    },
    "t_geri_donusum_prog": {
        "x_col": "tekstil_atik_kg",
        "x_label": "Tekstil atığı (kg)",
        "title": "Geri dönüşüm: Tekstil atığı ve azaltım ilişkisi",
        "profile_cols": [
            "parti_buyuklugu",
            "tekstil_atik_kg",
            "hammadde_ton",
            "kapasite_kullanim_pct",
        ],
    },
}


def _safe_karsio_label(t):
    return KARSIO_LABELS.get(t, str(t).replace("t_", "").replace("_", " "))


def _save_karsio_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(path)}")


if _ozet_df.empty or _full_parti_df.empty or _top10_full_df.empty:
    print("  ⚠ Karşıolgusal grafikler atlandı: gerekli DataFrame'lerden biri boş.")
else:


    fig11, ax11 = plt.subplots(figsize=(10, 6))
    fig11.patch.set_facecolor(COLORS["bg"])
    ax11.set_facecolor(COLORS["bg"])

    _plot_df = _ozet_df.copy()
    _plot_df["label"] = _plot_df["treatment_variable"].map(_safe_karsio_label)

    x = np.arange(len(_plot_df))
    width = 0.36

    ax11.bar(
        x - width / 2,
        _plot_df["mean_observed_outcome"],
        width=width,
        label="Mevcut ortalama outcome",
        color=COLORS["neutral"],
        alpha=0.85,
        edgecolor="white",
    )

    ax11.bar(
        x + width / 2,
        _plot_df["mean_counterfactual_outcome"],
        width=width,
        label="Karşıolgusal ortalama outcome",
        color=COLORS["secondary"],
        alpha=0.90,
        edgecolor="white",
    )

    for i, row in _plot_df.iterrows():
        y_val = max(row["mean_observed_outcome"], row["mean_counterfactual_outcome"])
        ax11.text(
            i,
            y_val * 1.04,
            f"Ort. azaltım\n{row['mean_reduction_amount']:.1f}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="#374151",
        )

    ax11.set_xticks(x)
    ax11.set_xticklabels(_plot_df["label"], fontsize=9)
    ax11.set_ylabel("Ortalama outcome (kgCO₂e)", fontsize=10)
    ax11.set_title(
        "Karşıolgusal Analiz — Mevcut Outcome vs Karşıolgusal Outcome",
        fontsize=12,
        fontweight="bold",
        pad=12,
    )
    ax11.grid(axis="y", alpha=0.25)
    ax11.legend(fontsize=9)

    _save_karsio_fig("/content/outputs/fig11_karsio_observed_vs_counterfactual.png")


    fig17, ax17 = plt.subplots(figsize=(9, 6))
    fig17.patch.set_facecolor(COLORS["bg"])
    ax17.set_facecolor(COLORS["bg"])

    _box_data = []
    _box_labels = []

    for _t in COUNTERFACTUAL_TREATMENTS:
        _vals = _full_parti_df.loc[
            _full_parti_df["treatment_variable"] == _t,
            "reduction_amount",
        ].astype(float).values

        _box_data.append(_vals)
        _box_labels.append(_safe_karsio_label(_t))

    bp17 = ax17.boxplot(
        _box_data,
        patch_artist=True,
        labels=_box_labels,
        showmeans=True,
    )

    for patch in bp17["boxes"]:
        patch.set_facecolor(COLORS["secondary"])
        patch.set_alpha(0.65)

    for median in bp17["medians"]:
        median.set_color("black")
        median.set_linewidth(2)

    for mean in bp17["means"]:
        mean.set_marker("D")
        mean.set_markerfacecolor(COLORS["primary"])
        mean.set_markeredgecolor("white")
        mean.set_markersize(6)

    ax17.set_ylabel("Azaltım miktarı (kgCO₂e)", fontsize=10)
    ax17.set_title(
        "Karşıolgusal Analiz — Parti Bazında Azaltım Dağılımı",
        fontsize=12,
        fontweight="bold",
        pad=12,
    )
    ax17.grid(axis="y", alpha=0.25)

    _save_karsio_fig("/content/outputs/fig17_karsio_reduction_distribution_boxplot.png")

print("\nKarşıolgusal görsel çıktıları:")
print("  /content/outputs/fig11_karsio_observed_vs_counterfactual.png")
print("  /content/outputs/fig17_karsio_reduction_distribution_boxplot.png")


print("\n" + "=" * 80)
print("TÜM ANALİZ TAMAMLANDI")
print("=" * 80)
print("\nNedensellik çıktıları:")
print("  /content/outputs/causal_forest_sonuclar.xlsx              (11 sayfa)")
print("    Sayfa  1: Ozet_Sonuclar")
print("    Sayfa  2: ITE_Satir_Bazli")
print("    Sayfa  3: Treatment_Outcome_Eslestirme")
print("    Sayfa  4: Feature_Importance")
print("    Sayfa  5: Placebo_Duyarlilik")
print("    Sayfa  6: CATE_Segment_Analizi")
print("    Sayfa  7: CATE_Quartile_Profil")
print("    Sayfa  8: CATE_Pivot_UrunxTreatment")
print("    Sayfa  9: CATE_Pivot_PartixTreatment")
print("    Sayfa 10: CATE_Karar_Destek_Ozeti")
print("    Sayfa 11: Segment_Esik_Degerleri")
print()
print("  /content/outputs/fig1_ate_forest_plot.png")
print("  /content/outputs/fig2_ite_distributions.png")
print("  /content/outputs/fig3_feature_importance.png")
print("  /content/outputs/fig4_ate_comparison.png")
print("  /content/outputs/fig5_cate_boxplots.png")
print("  /content/outputs/fig6_cate_segment_urun_kumas.png")
print("  /content/outputs/fig7_cate_segment_kapasite_makine.png")
print("  /content/outputs/fig8_cate_heatmap_urun_treatment.png")
print("  /content/outputs/fig9_cate_quartile_profil.png")
print("  /content/outputs/fig10_cate_heatmap_parti_treatment.png")
print()
print("Karşıolgusal analiz çıktıları:")
print("  /content/outputs/karsiolgusal_sonuclar.xlsx               (3 sayfa)")
print("    Sayfa 1: Karsio_Ozet         — treatment-outcome bazlı özet istatistikler")
print("    Sayfa 2: Karsio_Parti_Bazli  — parti bazlı karşıolgusal sonuçlar + profil")
print("    Sayfa 3: Karsio_Top10        — en yüksek azaltım potansiyelli top-10 partiler")
print()
print("Karşıolgusal görsel çıktıları:")
print("  /content/outputs/fig11_karsio_observed_vs_counterfactual.png")
print("  /content/outputs/fig17_karsio_reduction_distribution_boxplot.png")

1. VERİ YÜKLEME
Orijinal veri seti boyutu: (1000, 38)
Toplam eksik veri sayısı: 0

2. TREATMENT VE OUTCOME TANIMLARI
Confounder setleri oluşturuldu (temel + eş zamanlı treatmentlar).

3. SÜTUN KONTROLÜ
Tüm gerekli sütunlar veri setinde mevcut.

4. ANA CAUSAL FOREST MODELLERİ

▶ Yenilenebilir Enerji → Scope 2
  ATE = -235.9379 [-262.6610, -209.2149] | CI sıfırı içermiyor ✓

▶ Makine Modernizasyon → Scope 2
  ATE = -55.0972 [-70.9609, -39.2334] | CI sıfırı içermiyor ✓

▶ Yerel Tedarik → Scope 3 Taşıma
  ATE = -69.3608 [-426.0218, 287.3002] | CI sıfırı içeriyor

▶ Geri Dönüşüm Programı → Scope 3 Atık
  ATE = -70.7571 [-83.7969, -57.7173] | CI sıfırı içermiyor ✓

▶ Düşük Emisyonlu Boya → Scope 3 Kimyasal
  ATE = 30.7862 [-47.1980, 108.7704] | CI sıfırı içeriyor

▶ Parti Optimizasyon → Scope 2
  ATE = -1.3254 [-16.5568, 13.9059] | CI sıfırı içeriyor

5. PLACEBO TESTİ
  t_yenilenebilir_enerji           Placebo ATE = -5.9720
  t_makine_modernizasyon           Placebo ATE = 1.1815
  t_yerel_te